In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler,LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
df=pd.read_csv(r'C:\Users\ASUS\OneDrive\Desktop\TY-Dl-codes\train(1).csv')

In [3]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [8]:
df = df.drop(columns=["PassengerId","Name","Ticket","Cabin"])

In [10]:
df["Age"].fillna(df["Age"].median(), inplace=True)
df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_21576\1139536044.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Age"].fillna(df["Age"].median(), inplace=True)


In [12]:
df['Sex']=df['Sex'].map({'male':0,'female':1})
df['Embarked']=LabelEncoder().fit_transform(df['Embarked'])


In [16]:
X = df[["Pclass","Sex","Age","SibSp","Parch","Fare","Embarked"]].values
y = df["Survived"].values.reshape(-1,1)

In [17]:
X = MinMaxScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [18]:
def init_params():
    np.random.seed(42)
    return {
        "W1": np.random.randn(7,8)*0.1, "b1": np.zeros((1,8)),
        "W2": np.random.randn(8,4)*0.1, "b2": np.zeros((1,4)),
        "W3": np.random.randn(4,1)*0.1, "b3": np.zeros((1,1))
    }

In [19]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [20]:
def forwardprop(X, p):
    A1 = sigmoid(X @ p["W1"] + p["b1"])
    A2 = sigmoid(A1 @ p["W2"] + p["b2"])
    A3 = sigmoid(A2 @ p["W3"] + p["b3"])
    cache = (X, A1, A2, A3)
    return A3, cache

In [21]:
def backprop(y, p, cache, lr=0.01):
    X, A1, A2, A3 = cache
    m = X.shape[0]
    
    dZ3 = A3 - y
    dW3 = (A2.T @ dZ3) / m
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m
    
    dZ2 = (dZ3 @ p["W3"].T) * A2 * (1 - A2)
    dW2 = (A1.T @ dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m
    
    dZ1 = (dZ2 @ p["W2"].T) * A1 * (1 - A1)
    dW1 = (X.T @ dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m
    
    p["W3"] -= lr * dW3; p["b3"] -= lr * db3
    p["W2"] -= lr * dW2; p["b2"] -= lr * db2
    p["W1"] -= lr * dW1; p["b1"] -= lr * db1
    
    return p

In [22]:
def loss(y, y_hat):
    return -np.mean(
        y*np.log(y_hat+1e-8) + (1-y)*np.log(1-y_hat+1e-8)
    )

In [23]:
def predict(X, p):
    y_hat, _ = forwardprop(X, p)
    return (y_hat > 0.5).astype(int)

In [24]:
params = init_params()

for epoch in range(1000):
    y_hat, cache = forwardprop(X_train, params)
    params = backprop(y_train, params, cache)
    
    if epoch % 100 == 0:
        print("Epoch:", epoch, "Loss:", round(loss(y_train, y_hat), 4))

Epoch: 0 Loss: 0.7004
Epoch: 100 Loss: 0.6763
Epoch: 200 Loss: 0.6675
Epoch: 300 Loss: 0.6642
Epoch: 400 Loss: 0.663
Epoch: 500 Loss: 0.6625
Epoch: 600 Loss: 0.6623
Epoch: 700 Loss: 0.6622
Epoch: 800 Loss: 0.6622
Epoch: 900 Loss: 0.6622


In [25]:
train_acc = np.mean(predict(X_train, params) == y_train) * 100
test_acc = np.mean(predict(X_test, params) == y_test) * 100

print("Train Accuracy:", round(train_acc, 2), "%")
print("Test Accuracy:", round(test_acc, 2), "%")

Train Accuracy: 62.36 %
Test Accuracy: 58.66 %


In [26]:
print("Sample Predictions:")
preds = predict(X_test, params)

for i in range(5):
    print("Pred:", preds[i][0], "| Actual:", y_test[i][0])

Sample Predictions:
Pred: 0 | Actual: 1
Pred: 0 | Actual: 0
Pred: 0 | Actual: 0
Pred: 0 | Actual: 1
Pred: 0 | Actual: 1
